# BABEL

In [4]:
BABEL_PATH = r"/local/sqlite/BABEL.db"

level: str = "L1"
text_input = "syde2"

from sqlite_utils import Database
from collections import Counter
from typing import Optional
import time

db = Database(BABEL_PATH)
db.enable_wal()

prioritize: Optional[frozenset[str]] = frozenset(["OrganismTaxon", "Gene"])
prioritize_placeholders: Optional[str] = (
    ", ".join([f":prioritize{idx}" for idx in range(len(prioritize))])
    if prioritize
    else None
)

avoid: Optional[frozenset[str]] = (
    None  # frozenset(["OrganismTaxon", "BiologicalProcess"])
)
avoid_placeholders: Optional[str] = (
    ", ".join([f":avoid{idx}" for idx in range(len(avoid))]) if avoid else None
)

taxon: str = str(9606)

CategoryFrequency: Counter[str] = Counter()
CategoryFrequency["Gene"] += 1

top_logged_category: list[str] = CategoryFrequency.most_common(1)
most_common: Optional[str] = (
    str(top_logged_category[0][0]) if top_logged_category else None
)

sql_params: dict[str, str] = {"input": text_input.lower()}
if prioritize:
    sql_params.update(
        {f"prioritize{idx}": category for idx, category in enumerate(prioritize)}
    )
if avoid:
    sql_params.update({f"avoid{idx}": category for idx, category in enumerate(avoid)})
if taxon:
    sql_params["taxon"] = taxon
if most_common:
    sql_params["most_common"] = most_common

sql: str = f"""
    SELECT
        NAMES.CURIE,
        NAMES.CATEGORY,
        NAMES.NAME,
        NAMES.TAXON
    FROM SYNONYMS
    INNER JOIN NAMES ON SYNONYMS.CURIE = NAMES.CURIE
    WHERE 
        {"SYNONYMS.L1 = :input" if level == "L1" else "SYNONYMS.L2 = :input" if level == "L2" else "SYNONYMS.L3 = :input"}
        {"AND (NAMES.CATEGORY != 'Gene' OR NAMES.TAXON = :taxon)" if taxon else ""}
        {f"AND NAMES.CATEGORY NOT IN ({avoid_placeholders})" if avoid_placeholders else ""}
    {f"ORDER BY \n\t CASE \n\t\t WHEN NAMES.CATEGORY IN ({prioritize_placeholders}) AND NAMES.CATEGORY = :most_common THEN 0 \n\t\t WHEN NAMES.CATEGORY IN ({prioritize_placeholders}) THEN 1 \n\t\t WHEN NAMES.CATEGORY = :most_common THEN 2 \n\t\t ELSE 3 \n\t END" if prioritize_placeholders and most_common else f"ORDER BY \n\t CASE \n\t\t WHEN NAMES.CATEGORY IN ({prioritize_placeholders}) THEN 0 \\n\t\t ELSE 1 \n\t END" if prioritize_placeholders else "ORDER BY \n\t CASE \n\t\t WHEN NAMES.CATEGORY = :most_common THEN 0 \n\t\t ELSE 1 \n\t END" if most_common else ""}
    """

start = time.time()
rows = list(db.query(sql, sql_params))
print(time.time() - start)
print(sql)
print(sql_params)
print(rows)

4.47209095954895

    SELECT
        NAMES.CURIE,
        NAMES.CATEGORY,
        NAMES.NAME,
        NAMES.TAXON
    FROM SYNONYMS
    INNER JOIN NAMES ON SYNONYMS.CURIE = NAMES.CURIE
    WHERE 
        SYNONYMS.L1 = :input
        AND (NAMES.CATEGORY != 'Gene' OR NAMES.TAXON = :taxon)
        
    ORDER BY 
	 CASE 
		 WHEN NAMES.CATEGORY IN (:prioritize0, :prioritize1) AND NAMES.CATEGORY = :most_common THEN 0 
		 WHEN NAMES.CATEGORY IN (:prioritize0, :prioritize1) THEN 1 
		 WHEN NAMES.CATEGORY = :most_common THEN 2 
		 ELSE 3 
	 END
    
{'input': 'syde2', 'prioritize0': 'OrganismTaxon', 'prioritize1': 'Gene', 'taxon': '9606', 'most_common': 'Gene'}
[{'CURIE': 'NCBIGene:84144', 'CATEGORY': 'Gene', 'NAME': 'SYDE2', 'TAXON': 9606}, {'CURIE': 'PR:000015867', 'CATEGORY': 'Protein', 'NAME': 'Rho GTPase-activating protein SYDE2', 'TAXON': ''}, {'CURIE': 'UniProtKB:Q5VT97', 'CATEGORY': 'Protein', 'NAME': 'SYDE2_HUMAN Rho GTPase-activating protein SYDE2 (sprot)', 'TAXON': 9606}]


# KG2

In [2]:
KG2_PATH = r"/ssd/sqlite/KG2.10.1.db"

db2 = Database(KG2_PATH)
db2.enable_wal()

sql: str = f"""
    SELECT
        clusters.cluster_id,
        clusters.category,
        clusters.name
    FROM nodes
    INNER JOIN clusters ON nodes.cluster_id = clusters.cluster_id
    WHERE
        {"nodes.name = :input" if level == "L1" else "nodes.name_simplified = :input"}
        {f"AND clusters.category NOT IN ({avoid_placeholders})" if avoid_placeholders else ""}
    {f"ORDER BY \n\t CASE \n\t\t WHEN clusters.category IN ({prioritize_placeholders}) AND clusters.category = :most_common THEN 0 \n\t\t WHEN clusters.category IN ({prioritize_placeholders}) THEN 1 \n\t\t WHEN clusters.category = :most_common THEN 2 \n\t\t ELSE 3 \n\t END" if prioritize_placeholders and most_common else f"ORDER BY \n\t CASE \n\t\t WHEN clusters.category IN ({prioritize_placeholders}) THEN 0 \\n\t\t ELSE 1 \n\t END" if prioritize_placeholders else "ORDER BY \n\t CASE \n\t\t WHEN clusters.category = :most_common THEN 0 \n\t\t ELSE 1 \n\t END" if most_common else ""}
    """

sql_params["input"] = text_input

start = time.time()
rows = list(db2.query(sql, sql_params))
print(time.time() - start)
print(sql)
print(sql_params)
print(rows)

0.003498077392578125

    SELECT
        clusters.cluster_id,
        clusters.category,
        clusters.name
    FROM nodes
    INNER JOIN clusters ON nodes.cluster_id = clusters.cluster_id
    WHERE
        nodes.name = :input
        AND clusters.category NOT IN (:avoid0, :avoid1)
    ORDER BY 
	 CASE 
		 WHEN clusters.category IN (:prioritize0) AND clusters.category = :most_common THEN 0 
		 WHEN clusters.category IN (:prioritize0) THEN 1 
		 WHEN clusters.category = :most_common THEN 2 
		 ELSE 3 
	 END
    
{'input': 'galactose', 'prioritize0': 'SmallMolecule', 'avoid0': 'OrganismTaxon', 'avoid1': 'BiologicalProcess', 'taxon': '9606', 'most_common': 'disease'}
[{'cluster_id': 'CHEBI:28061', 'category': 'SmallMolecule', 'name': 'alpha-D-galactose'}, {'cluster_id': 'CHEBI:4139', 'category': 'SmallMolecule', 'name': 'D-galactopyranose'}, {'cluster_id': 'CHEBI:4139', 'category': 'SmallMolecule', 'name': 'D-galactopyranose'}, {'cluster_id': 'CHEBI:4139', 'category': 'SmallMolecule', 

# METADATA

In [6]:
from sqlite_utils import Database
import time

PUBMED_PATH: str = "/ssd/sqlite/PubMed.db"

db3 = Database(PUBMED_PATH)
db3.enable_wal()

sql: str = """
SELECT
    mesh.mesh_major,
    mesh.mesh,
    info.firstauthor,
    info.journal,
    info.title,
    info.year
FROM ids
INNER JOIN mesh ON ids.pmid = mesh.pmid
INNER JOIN info ON ids.pmid = info.pmid
WHERE ids.alt = :curie OR ids.pmid = :curie
"""

# NO CURIE PREFIX
start = time.time()
sql_input = {"curie": "39264809"}
rows = list(db3.query(sql, sql_input))
print(time.time() - start)
print(sql)
print(sql_input)
print(rows)

0.0010099411010742188

SELECT
    mesh.mesh_major,
    mesh.mesh,
    info.firstauthor,
    info.journal,
    info.title,
    info.year
FROM ids
INNER JOIN mesh ON ids.pmid = mesh.pmid
INNER JOIN info ON ids.pmid = info.pmid
WHERE ids.alt = :curie OR ids.pmid = :curie

{'curie': '39264809'}
[{'mesh_major': 'N', 'mesh': 'D006801', 'firstauthor': 'Boverhoff', 'journal': 'Cell reports', 'title': 'Profiling the fecal microbiome and its modulators across the lifespan in the Netherlands.', 'year': 2024}, {'mesh_major': 'Y', 'mesh': 'D005243', 'firstauthor': 'Boverhoff', 'journal': 'Cell reports', 'title': 'Profiling the fecal microbiome and its modulators across the lifespan in the Netherlands.', 'year': 2024}, {'mesh_major': 'N', 'mesh': 'D000328', 'firstauthor': 'Boverhoff', 'journal': 'Cell reports', 'title': 'Profiling the fecal microbiome and its modulators across the lifespan in the Netherlands.', 'year': 2024}, {'mesh_major': 'N', 'mesh': 'D000368', 'firstauthor': 'Boverhoff', 'journa

# PMCCAPTIONS

In [12]:
PMC_PATH: str = "/ssd/sqlite/PMCSuppCaptions.db"

db4 = Database(PMC_PATH)
db4.enable_wal()

sql: str = """
SELECT caption
FROM captions
WHERE pmc = :curie AND file = :filename
LIMIT 1
"""

# NO PMC AT ALL
start = time.time()
sql_input = {"curie": "7562722", "filename": "41467_2020_18871_MOESM4_ESM.xlsx"}
rows = list(db4.query(sql, sql_input))
print(time.time() - start)
print(sql)
print(sql_input)
print(rows)

0.0013835430145263672

SELECT caption
FROM captions
WHERE pmc = :curie AND file = :filename
LIMIT 1

{'curie': '7562722', 'filename': '41467_2020_18871_MOESM4_ESM.xlsx'}
[{'caption': 'Supplementary Data 1-9'}]
